AutoML_DecisionTrees/
│
├── core/
│   ├── DecisionTreeDecoder.py       #  Builds actual decision trees from genes
│   ├── DT_genes.py                  #  Encodes genetic mappings for tree components
│   ├── SplitCriteria.py             #  Implements split quality measures
│   ├── StoppingCriteria.py          #  Defines stopping rules for tree growth
│   ├── MissingValues.py             #  Handles missing values during split & prediction
│   ├── PruningStrategies.py         #  Implements post-pruning algorithms
│   ├── Node.py                      #  Defines the basic tree node structure
│   ├── metrics.py                   #  Evaluation utilities (F1, recall, specificity)
│   └── utils/                       #  Future helper functions
│
├── notebooks/
│   ├── demo_decision_tree.ipynb     #   Example notebook 
│
├── data/                            #   CSVs or sklearn datasets
│
└── README.md                        #   Project overview and usage instructions


In [1]:
import sys
from pathlib import Path

# Determine the project root directory (two levels up from current file)
PROJECT_ROOT = Path.cwd().parents[1]

# Add to Python path 
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root added:", PROJECT_ROOT)


Project root added: c:\My_Files\Projects\AutoML_Drd


In [ ]:
# ============================================================
# Example Usage: Genetic Decision Tree System
# ============================================================

# Imports from local project
from core.DecisionTreeDecoder import GeneDecoder
from core.metrics import evaluate_tree
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import pandas as pd

# Load a dataset
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

#  Split data into train/test 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

#  Define a gene configuration
# Each integer refers to a strategy defined in DT_genes.py
gene = {
    "split": 0,     # "information_gain"
    "stop": 2,         # "min_samples_split"
    "stop_param": 1,   # small -> less restrictive
    "mv_split": 0,     # "ignore_all" (simplify handling)
    "mv_distrib": 0,   # "ignore_all"
    "mv_classif": 0,   # "explore_all"
    "prune": 0,        # None (no pruning)
}

#  Decode gene into operational Decision Tree 
decoder = GeneDecoder(gene)
tree = decoder.decode(X_train)

#  Train tree on training data 
tree.fit(X_train, y_train)

#  Evaluate performance on test set 
#! evaluate_tree calls .fit again
metrics_result = evaluate_tree(tree, X_train, y_train, X_test, y_test)
print("\n Evaluation Metrics:")
for k, v in metrics_result.items():
    if v is None:
        print(f"{k}: {v}")
    else:
        print(f"{k}: {v:.4f}")

#  Visualize tree structure 
print("\n Tree Structure:")
tree.print_tree()


 Evaluation Metrics:
accuracy: 0.9333
balanced_accuracy: 0.9333
precision: 0.9345
recall: 0.9333
f1: 0.9333
specificity: 0.9667
mcc: 0.9007
n_nodes: 15.0000
depth: None

 Tree Structure:
[Xpetal length (cm) <= 1.900] gain=0.9183
  Leaf -> Class 0 (35 samples)
  [Xpetal width (cm) <= 1.500] gain=0.7478
    [Xpetal length (cm) <= 4.900] gain=0.1914
      Leaf -> Class 1 (33 samples)
      Leaf -> Class 2 (1 samples)
    [Xsepal length (cm) <= 6.000] gain=0.1293
      [Xsepal length (cm) <= 5.800] gain=0.4669
        Leaf -> Class 2 (5 samples)
        [Xsepal length (cm) <= 5.900] gain=0.2516
          [Xsepal width (cm) <= 3.000] gain=1.0000
            Leaf -> Class 2 (1 samples)
            Leaf -> Class 1 (1 samples)
          Leaf -> Class 1 (1 samples)
      Leaf -> Class 2 (28 samples)


Check usability in a range of datasets 

In [3]:
# Liste des datasets UCI utilisés dans HEAD-DT / MOHEAD-DT
UCI_DATASETS = {    
    "abalone": "https://archive.ics.uci.edu/ml/machine-learning-databases/abalone/abalone.data",
    "anneal": "https://archive.ics.uci.edu/ml/machine-learning-databases/annealing/anneal.data",
    "arrhythmia": "https://archive.ics.uci.edu/ml/machine-learning-databases/arrhythmia/arrhythmia.data",    
    "bridges-version1": "https://archive.ics.uci.edu/ml/machine-learning-databases/bridges/bridges.data.version1",
    "car": "https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data",
    "cylinder-bands": "https://archive.ics.uci.edu/ml/machine-learning-databases/cylinder-bands/bands.data",
    "glass": "https://archive.ics.uci.edu/ml/machine-learning-databases/glass/glass.data",
    "hepatitis": "https://archive.ics.uci.edu/ml/machine-learning-databases/hepatitis/hepatitis.data",
    "iris": "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data",
    "semeion": "https://archive.ics.uci.edu/ml/machine-learning-databases/semeion/semeion.data",
    "shuttle_landing": "https://archive.ics.uci.edu/ml/machine-learning-databases/shuttle-landing-control/shuttle-landing-control.data",
    "winequality-red": "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
    "winequality-white": "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv",
    # ➕ Tu peux ajouter ici d'autres datasets de Table 1
}
# "audiology": "https://archive.ics.uci.edu/ml/machine-learning-databases/audiology/audiology.data",
# "segment": "https://archive.ics.uci.edu/ml/machine-learning-databases/image/segmentation.data",

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [5]:
def load_uci_dataset(name, url):
    # Handle different separators 
    sep = ";" if "winequality" in name else ","

    df = pd.read_csv(
        url,
        header=None,
        sep=sep,
        na_values=["?"]
    )

    #df = df.dropna(how="all")

    # I assumed last column as target ALWAYS
    X = df.iloc[:, :-1]
    y = df.iloc[:, -1]

    # Encode target labels
    y = LabelEncoder().fit_transform(y.astype(str))

    return X, y


In [6]:
BASELINE_GENE = {
    "split": 0,        # information_gain
    "stop": 2,         # min_samples_split
    "stop_param": 5,
    "mv_split": 0,     # ignore_all
    "mv_distrib": 0,
    "mv_classif": 0,
    "prune": 0
}


In [7]:
BAD_BASELINE_GENE = {
    "split": 0,          # information_gain (aggressive)
    "stop": 2,           # min_samples_split
    "stop_param": 1,     # allows splitting with 1 sample
    "mv_split": 0,       # ignore_all
    "mv_distrib": 0,
    "mv_classif": 0,
    "prune": 0
}

In [8]:
from core.DecisionTreeDecoder import GeneDecoder

results = []

for name, url in UCI_DATASETS.items():
    print(f"\n Dataset in test: {name}")

    try:
        X, y = load_uci_dataset(name, url)

        print("Missing values in dataset:", X.isnull().sum().sum())

        # Basic sanity check
        if X.shape[0] < 20:
            raise ValueError("Dataset too small")

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42
        )

        print("Missing in train:", X_train.isnull().sum().sum())
        print("Missing in test:", X_test.isnull().sum().sum())

        decoder = GeneDecoder(BASELINE_GENE)
        tree = decoder.decode(X_train)

        tree.fit(X_train, y_train)

        print(f"Tree built successfully with nodes: {tree.count_nodes()}")

        results.append({
            "dataset": name,
            "status": "OK",
            "n_samples": X.shape[0],
            "n_features": X.shape[1],
            "n_nodes": tree.count_nodes()
        })

    except Exception as e:
        print(f" Failed: {e}")

        results.append({
            "dataset": name,
            "status": "FAIL",
            "error": str(e)
        })



 Dataset in test: abalone
Missing values in dataset: 0
Missing in train: 0
Missing in test: 0
Tree built successfully with nodes: 3065

 Dataset in test: anneal
Missing values in dataset: 19692
Missing in train: 13803
Missing in test: 5889
Tree built successfully with nodes: 7

 Dataset in test: arrhythmia
Missing values in dataset: 408
Missing in train: 279
Missing in test: 129
Tree built successfully with nodes: 35

 Dataset in test: bridges-version1
Missing values in dataset: 75
Missing in train: 50
Missing in test: 25
Tree built successfully with nodes: 41

 Dataset in test: car
Missing values in dataset: 0
Missing in train: 0
Missing in test: 0
Tree built successfully with nodes: 147

 Dataset in test: cylinder-bands
Missing values in dataset: 1037
Missing in train: 671
Missing in test: 366
Tree built successfully with nodes: 91

 Dataset in test: glass
Missing values in dataset: 0
Missing in train: 0
Missing in test: 0
Tree built successfully with nodes: 11

 Dataset in test: he

In [9]:
results_df = pd.DataFrame(results)
display(results_df)

,dataset,status,n_samples,n_features,n_nodes,error
0,abalone,OK,4177.0,8.0,3065.0,NaN
1,anneal,OK,798.0,38.0,7.0,NaN
2,arrhythmia,OK,452.0,279.0,35.0,NaN
3,bridges-version1,OK,108.0,12.0,41.0,NaN
4,car,OK,1728.0,6.0,147.0,NaN
5,cylinder-bands,OK,541.0,39.0,91.0,NaN
6,glass,OK,214.0,10.0,11.0,NaN
7,hepatitis,OK,155.0,19.0,37.0,NaN
8,iris,OK,150.0,4.0,19.0,NaN
9,semeion,OK,1593.0,0.0,1.0,NaN


# New metrics

In [10]:
from core.DecisionTreeDecoder import GeneDecoder 
from core.metrics import evaluate_tree
from sklearn.model_selection import train_test_split
import numpy as np 

results = []

for name, url in UCI_DATASETS.items():
    print(f"\n Dataset in test: {name}")

    try:
        X, y = load_uci_dataset(name, url)

        print("Missing values in dataset:", X.isnull().sum().sum())

        # Basic sanity check
        if X.shape[0] < 20:
            raise ValueError("Dataset too small")

        # train test and splitting with strtify only if possible 

        unique, counts = np.unique(y, return_counts=True)
        min_class_count = counts.min()

        if min_class_count < 2:
            print("Stratified split not possible — using regular split.")
            X_train, X_test, y_train, y_test = train_test_split(
                X, y,
                test_size=0.3,
                random_state=42
            )
        else:
            X_train, X_test, y_train, y_test = train_test_split(
                X, y,
                test_size=0.3,
                random_state=42,
                stratify=y
            )

        print("Missing in train:", X_train.isnull().sum().sum())
        print("Missing in test:", X_test.isnull().sum().sum())

        decoder = GeneDecoder(BASELINE_GENE)
        tree = decoder.decode(X_train)

        tree.fit(X_train, y_train)

        print(f"Tree built successfully with nodes: {tree.count_nodes()}")

        # ADD EVALUATION HERE
        metrics = evaluate_tree(tree, X_train, y_train, X_test, y_test)

        # Append everything to results
        results.append({
            "dataset": name,
            "status": "OK",
            "n_samples": X.shape[0],
            "n_features": X.shape[1],
            **metrics   # expands accuracy, f1, mcc, etc.
        })

    except Exception as e:
        print(f" Failed: {e}")

        results.append({
            "dataset": name,
            "status": "FAIL",
            "error": str(e)
        })


 Dataset in test: abalone
Missing values in dataset: 0
Stratified split not possible — using regular split.
Missing in train: 0
Missing in test: 0
Tree built successfully with nodes: 3065


c:\Users\sarah\anaconda3\envs\EvoML\Lib\site-packages\sklearn\metrics\_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")



 Dataset in test: anneal
Missing values in dataset: 19692
Missing in train: 13784
Missing in test: 5908
Tree built successfully with nodes: 5

 Dataset in test: arrhythmia
Missing values in dataset: 408
Missing in train: 291
Missing in test: 117
Tree built successfully with nodes: 37

 Dataset in test: bridges-version1
Missing values in dataset: 75
Stratified split not possible — using regular split.
Missing in train: 50
Missing in test: 25
Tree built successfully with nodes: 41

 Dataset in test: car
Missing values in dataset: 0
Missing in train: 0
Missing in test: 0
Tree built successfully with nodes: 145

 Dataset in test: cylinder-bands
Missing values in dataset: 1037
Missing in train: 669
Missing in test: 368
Tree built successfully with nodes: 97

 Dataset in test: glass
Missing values in dataset: 0
Missing in train: 0
Missing in test: 0
Tree built successfully with nodes: 11

 Dataset in test: hepatitis
Missing values in dataset: 167
Missing in train: 131
Missing in test: 36
Tr

c:\Users\sarah\anaconda3\envs\EvoML\Lib\site-packages\sklearn\metrics\_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


Missing values in dataset: 0
 Failed: Dataset too small

 Dataset in test: winequality-red
Missing values in dataset: 0
Stratified split not possible — using regular split.
Missing in train: 0
Missing in test: 0
Tree built successfully with nodes: 577

 Dataset in test: winequality-white
Missing values in dataset: 0
Stratified split not possible — using regular split.
Missing in train: 0
Missing in test: 0
Tree built successfully with nodes: 1793


In [11]:
results_df = pd.DataFrame(results)
display(results_df)

,dataset,status,n_samples,n_features,accuracy,balanced_accuracy,precision,recall,f1,specificity,mcc,n_nodes,depth,error
0,abalone,OK,4177.0,8.0,0.204147,0.130504,0.120821,0.119156,0.116916,0.961217,0.108956,3065.0,NaN,NaN
1,anneal,OK,798.0,38.0,0.112500,0.200000,0.022500,0.200000,0.040449,0.800000,0.000000,5.0,NaN,NaN
2,arrhythmia,OK,452.0,279.0,0.367647,0.129655,0.162821,0.129655,0.143333,0.930954,0.095859,37.0,NaN,NaN
3,bridges-version1,OK,108.0,12.0,0.727273,0.585165,0.601832,0.585165,0.552512,0.952535,0.650138,41.0,NaN,NaN
4,car,OK,1728.0,6.0,0.976879,0.963657,0.951383,0.963657,0.955696,0.991941,0.950572,145.0,NaN,NaN
5,cylinder-bands,OK,541.0,39.0,0.742331,0.490092,0.490267,0.490092,0.489794,0.822222,0.470587,97.0,NaN,NaN
6,glass,OK,214.0,10.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,11.0,NaN,NaN
7,hepatitis,OK,155.0,19.0,0.617021,0.603480,0.610784,0.603480,0.602444,0.603480,0.214140,27.0,NaN,NaN
8,iris,OK,150.0,4.0,0.933333,0.933333,0.934524,0.933333,0.933259,0.966667,0.900667,15.0,NaN,NaN
9,semeion,OK,1593.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.997912,0.000000,1.0,NaN,NaN
